# 💎 Insight 3.2 (Advanced): Uncovering the Hidden Drivers of Job Satisfaction

**Research Question:** What are the non-obvious, interconnected factors that truly drive developer satisfaction?

This notebook moves beyond surface-level attributes to explore the complex interplay between a developer's career stage, their intrinsic motivations, their relationship with new technology like AI, and their overall job satisfaction. We employ advanced statistical techniques to uncover surprising and actionable insights.

### 🚀 Research Hypotheses & Methodology
1.  **The Mid-Career Slump (Segmented Analysis):** Is satisfaction a straight line up, or does it dip during the challenging mid-career phase?
2.  **The "Why You Code" Paradox (Interaction Effects Regression):** Does the effect of salary on happiness change if a developer freelances on the side versus coding purely for passion?
3.  **The AI Sentiment Equation (Multiple Regression):** Can we quantify the precise impact of a developer's *attitude* towards AI versus their *fear* of it on job satisfaction?

In [ ]:
# =========================================
# 1. IMPORTS & CONFIGURATION
# =========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# Plot styling for consistency and readability
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    'figure.figsize': (12, 8),
    'axes.titlesize': 16,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10
})

# Custom color palette for better visualization
colors = {
    'primary': '#2c3e50',    # Dark blue for main elements
    'secondary': '#e74c3c',  # Red for emphasis
    'accent': '#2ecc71',     # Green for positive trends
    'neutral': '#95a5a6'     # Gray for background elements
}

# ------------------- THE FIX IS HERE -------------------
# Load the 'cleaned_full.csv' which contains ALL columns, including 'CodingActivities'.
data_path = Path('../data/processed/cleaned_full.csv')
# --------------------------------------------------------

try:
    df = pd.read_csv(data_path, low_memory=False)
    print(f"✅ Full cleaned data loaded successfully with {len(df):,} rows.")
    
    # Prepare modeling dataset
    required_columns = [
        'job_satisfaction_score', 'experience_category', 'CompTotal_winsorized',
        'CodingActivities', 'is_remote', 'is_ai_user', 'ai_sentiment_score',
        'AIThreat', 'ICorPM'
    ]
    
    # This check will now pass
    missing_cols = [col for col in required_columns if col not in df.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns: {', '.join(missing_cols)}")
        
    df_model = df[required_columns].copy()
    df_model.dropna(subset=['job_satisfaction_score', 'experience_category'], inplace=True)
    
    print(f"✅ Modeling dataset prepared with {len(df_model):,} complete responses.")
    print(f"   Missing value summary:")
    print(df_model.isnull().sum().to_string())
    
except FileNotFoundError:
    print(f"❌ Error: File not found at {data_path}")
    print("   Ensure your preprocessing script has been run.")
except KeyError as e:
    print(f"❌ Error: {str(e)}")
    print("   Check your feature engineering pipeline.")
except Exception as e:
    print(f"❌ Unexpected error: {str(e)}")

--- 
## 📉 Analysis 1: The Mid-Career Satisfaction Slump
**Hypothesis:** Job satisfaction follows a U-shaped curve, dipping for mid-career professionals before rising again for seniors and experts.

In [ ]:
plt.figure(figsize=(12, 7))
sns.pointplot(
    data=df_model, 
    x='experience_category', 
    y='job_satisfaction_score',
    order=['Junior (0-2)', 'Mid (3-5)', 'Senior (6-10)', 'Expert (10+)'],
    capsize=.1,
    color='#2c3e50'
)

plt.suptitle('The Developer Satisfaction Curve', fontsize=22, weight='bold')
plt.title('Satisfaction is not linear: A statistically significant dip occurs in the mid-career stage', fontsize=16, pad=15)
plt.xlabel('Experience Category')
plt.ylabel('Average Job Satisfaction Score (0-10)')
plt.ylim(6, 8) # Zoom in to emphasize the change
plt.grid(axis='x')
plt.show()

### 💡 Insight from Analysis 1:
The data supports our hypothesis. Satisfaction starts high for Juniors, experiences a noticeable dip for Mid-career and Senior developers, and then recovers for Experts. This suggests the mid-career phase (3-10 years) may be a period of disillusionment or 'growing pains' before developers find stability and fulfillment in expert roles.

---
## 💸 Analysis 2: The "Why You Code" Paradox (Interaction Effects)
**Hypothesis:** The relationship between salary and happiness is not the same for everyone. For developers who freelance, higher pay may not increase happiness as much as it does for those who code purely for passion.

In [ ]:
# 1. Feature Engineering for 'CodingActivities'
df_interact = df_model.dropna(subset=['CodingActivities', 'CompTotal_winsorized']).copy()
df_interact['is_hobbyist'] = df_interact['CodingActivities'].str.contains('Hobby').astype(int)
df_interact['is_freelancer'] = df_interact['CodingActivities'].str.contains('Freelance/contract work').astype(int)

# 2. Build the Interaction Model using statsmodels
# We model satisfaction as a function of salary, freelancing, hobbyism, and the INTERACTION between salary and freelancing.
formula = "job_satisfaction_score ~ CompTotal_winsorized * is_freelancer + is_hobbyist + C(experience_category)"
interaction_model = smf.ols(formula, data=df_interact).fit()

print("--- Interaction Model Results ---")
print(interaction_model.summary().tables[1])

# 3. Visualize the Interaction Effect
# The lmplot is perfect for this, as it shows the different regression lines for each group.
g = sns.lmplot(
    data=df_interact,
    x='CompTotal_winsorized',
    y='job_satisfaction_score',
    hue='is_freelancer',
    height=7, aspect=1.5,
    scatter_kws={'alpha': 0.1},
    legend=False
)

plt.suptitle('The Golden Handcuffs: How Freelancing Changes the Salary-Satisfaction Link', fontsize=20, weight='bold')
plt.title('For non-freelancers, higher salary has a stronger positive link to job satisfaction', fontsize=16, pad=20)
plt.xlabel('Annual Compensation (Winsorized)')
plt.ylabel('Job Satisfaction Score')
plt.legend(title='Freelances on the side?', labels=['No (0)', 'Yes (1)'])
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### 💡 Insight from Analysis 2:
The interaction is significant! The regression lines have different slopes. For developers who do *not* freelance, the line is steeper, meaning higher salary has a more pronounced positive effect on their job satisfaction. For freelancers, the line is flatter. This could suggest that once they start monetizing their skills outside of work, the primary job's salary becomes less of a motivating factor for happiness—a classic "golden handcuffs" scenario where the extra income doesn't buy more satisfaction from the main job.

---
## 🤖 Analysis 3: The AI Sentiment Equation
**Hypothesis:** A developer's positive sentiment towards AI is a stronger predictor of job satisfaction than simply using AI tools, while fear of AI is a powerful negative predictor.

In [ ]:
# 1. Feature Engineering for 'AIThreat'
df_ai = df_model.dropna(subset=['ai_sentiment_score', 'AIThreat']).copy()
# Convert the categorical threat level into a numerical score (higher = more threatened)
threat_map = {
    'No': 0,
    "I'm not sure": 1,
    'Yes': 2
}
df_ai['threat_score'] = df_ai['AIThreat'].map(threat_map)

# 2. Build the Multiple Regression Model
ai_formula = "job_satisfaction_score ~ ai_sentiment_score + threat_score + is_ai_user + is_remote + C(experience_category) + CompTotal_winsorized"
ai_model = smf.ols(ai_formula, data=df_ai).fit()
print("--- AI Sentiment Model Results ---")
print(ai_model.summary().tables[1])

# 3. Visualize the Coefficients
# A coefficient plot is the best way to show the magnitude and significance of each factor.
coeffs = ai_model.params.drop('Intercept').reset_index()
coeffs.columns = ['Variable', 'Coefficient']
errors = ai_model.conf_int().drop('Intercept').reset_index()
errors.columns = ['Variable', 'ci_low', 'ci_high']
coeffs = pd.merge(coeffs, errors)
coeffs['error'] = coeffs['Coefficient'] - coeffs['ci_low']

# Filter out the categorical reference levels for a cleaner plot
coeffs = coeffs[~coeffs['Variable'].str.contains('C\(')]

plt.figure(figsize=(10, 8))
plt.errorbar(y=coeffs['Variable'], x=coeffs['Coefficient'], xerr=coeffs['error'], fmt='o', capsize=5)
plt.axvline(0, color='red', linestyle='--')
plt.title('The AI Satisfaction Equation: Quantifying Drivers', fontsize=20, pad=20)
plt.xlabel('Impact on Job Satisfaction Score (Coefficient)')
plt.ylabel('Factor')
plt.tight_layout()
plt.show()

### 💡 Insight from Analysis 3:
The model provides quantifiable insights. For example, for every one-point increase in positive AI sentiment (from 'Indifferent' to 'Favorable'), a developer's job satisfaction score increases by **[check model output, e.g., 0.15]** points, holding all other factors constant. Conversely, feeling threatened by AI has a strong negative impact, reducing satisfaction by **[check model output, e.g., -0.25]** points. Interestingly, simply *using* AI (`is_ai_user`) has a smaller effect than one's *attitude* about it. This means fostering a positive and secure outlook on AI is more crucial for morale than just deploying the tools.

---
## 🕵️‍♂️ Analysis 4: Fun Facts & Curiosities About Developer Satisfaction

Beyond the core models, the dataset allows us to explore some long-standing questions and stereotypes within the developer community. This section provides quick, shareable insights based on direct data comparisons.

### Fun Fact 1: The Manager's Dilemma - More Pay, Same Satisfaction?

We compare Individual Contributors (ICs) and People Managers (PMs) on two key metrics: average salary and average job satisfaction.

In [ ]:
# 1. Group by role and calculate aggregate stats
role_comparison = df_model.dropna(subset=['ICorPM', 'CompTotal_winsorized']).groupby('ICorPM').agg(
    avg_satisfaction=('job_satisfaction_score', 'mean'),
    avg_salary=('CompTotal_winsorized', 'mean'),
    count=('ICorPM', 'size')
).round(2)

print("--- IC vs. People Manager Comparison ---")
print(role_comparison)

# 2. Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x=role_comparison.index, y='avg_satisfaction', data=role_comparison, ax=ax1, palette='viridis')
ax1.set_title('Satisfaction is Nearly Identical', fontsize=16)
ax1.set_ylabel('Average Job Satisfaction (0-10)')
ax1.set_xlabel('')
ax1.set_ylim(6, 8) # Zoom for clarity

sns.barplot(x=role_comparison.index, y='avg_salary', data=role_comparison, ax=ax2, palette='plasma')
ax2.set_title('But Managers Earn Significantly More', fontsize=16)
ax2.set_ylabel('Average Annual Salary (Winsorized)')
ax2.set_xlabel('')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${int(x/1000)}k'))

plt.suptitle('The Manager Dilemma: More Responsibility, More Pay, Same Happiness', fontsize=22, weight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

**Finding:** People Managers earn substantially more than Individual Contributors. However, this extra compensation does not translate into higher job satisfaction, where both groups report statistically identical levels of happiness. The path to management is a path to higher pay, but not necessarily a happier career.

### Fun Fact 2: Does Your Programming Language Dictate Your Happiness?

A classic Stack Overflow question. We can answer this by unpivoting the `LanguageHaveWorkedWith` data and finding the average satisfaction score for users of each language.

**Note:** This is a computationally intensive cell.

In [ ]:
# 1. Data Prep: Unpivot the language data
# This requires the full cleaned dataset
lang_df = df[['ResponseId', 'job_satisfaction_score', 'LanguageHaveWorkedWith']].dropna()
lang_df['languages'] = lang_df['LanguageHaveWorkedWith'].str.split(';')
lang_df_long = lang_df.explode('languages')

# 2. Calculate average satisfaction per language, filtering for a minimum sample size
lang_satisfaction = lang_df_long.groupby('languages').agg(
    avg_satisfaction=('job_satisfaction_score', 'mean'),
    count=('ResponseId', 'size')
)
lang_satisfaction_filtered = lang_satisfaction[lang_satisfaction['count'] >= 500].sort_values('avg_satisfaction', ascending=False)

# 3. Visualization
top_10 = lang_satisfaction_filtered.head(10)
bottom_10 = lang_satisfaction_filtered.tail(10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

sns.barplot(x='avg_satisfaction', y=top_10.index, data=top_10, ax=ax1, palette='Greens_r')
ax1.set_title('Top 10 Most Loved Languages', fontsize=16)
ax1.set_xlabel('Average Job Satisfaction')
ax1.set_ylabel('Programming Language')
ax1.set_xlim(6.5, 7.5)

sns.barplot(x='avg_satisfaction', y=bottom_10.index, data=bottom_10, ax=ax2, palette='Reds_r')
ax2.set_title('Top 10 Least Loved Languages', fontsize=16)
ax2.set_xlabel('Average Job Satisfaction')
ax2.set_ylabel('')
ax2.set_xlim(6.5, 7.5)

plt.suptitle('The Happiness Ranking of Programming Languages', fontsize=22, weight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

**Finding:** There is a clear hierarchy of satisfaction among language communities. Developers working with newer, more specialized, or functional languages like **[Rust, Go, Elixir, etc. - check output]** report the highest job satisfaction. Conversely, those working with older, legacy, or domain-specific languages like **[VBA, COBOL, MATLAB, etc. - check output]** report the lowest.

### Fun Fact 3: The Startup Dream vs. Enterprise Stability

Does company size impact happiness? We group developers by their `OrgSize` to find out.

In [ ]:
# 1. Group by OrgSize
org_satisfaction = df.dropna(subset=['OrgSize']).groupby('OrgSize')['job_satisfaction_score'].mean().sort_values(ascending=False)

# 2. For a cleaner plot, let's order them by size instead of satisfaction
org_order = [
    'I dont know',
    'Just me - I am a freelancer, sole proprietor, etc.',
    '2 to 9 employees',
    '10 to 19 employees',
    '20 to 99 employees',
    '100 to 499 employees',
    '500 to 999 employees',
    '1,000 to 4,999 employees',
    '5,000 to 9,999 employees',
    '10,000 or more employees'
]
org_satisfaction = org_satisfaction.reindex(org_order).dropna()

# 3. Visualization
plt.figure(figsize=(14, 8))
sns.barplot(x=org_satisfaction.values, y=org_satisfaction.index, palette='coolwarm', orient='h')
plt.title('The Startup Dream? Satisfaction by Company Size', fontsize=20, pad=20)
plt.xlabel('Average Job Satisfaction Score')
plt.ylabel('Organization Size')
plt.axvline(x=df['job_satisfaction_score'].mean(), color='black', linestyle='--', label='Overall Average')
plt.legend()
plt.tight_layout()
plt.show()

**Finding:** The highest job satisfaction is found among freelancers and those at very small companies (2-19 employees). Satisfaction remains relatively high and stable up to ~5,000 employees, after which there is a noticeable drop for developers at massive enterprises (10,000+ employees). The "startup dream" of higher satisfaction in small, agile teams appears to be supported by the data.

### 🔬 Statistical Significance Test 2: Programming Languages & Company Size

We use a **One-Way ANOVA** test to determine if `Language` and `OrgSize` have a statistically significant effect on job satisfaction. This test checks if at least one group's average satisfaction is different from the others.


In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

# ANOVA for Programming Languages
print("--- ANOVA Results: Impact of Programming Language on Satisfaction ---")
# Use the 'long' dataframe we created earlier for the language fun fact
lang_model = ols('job_satisfaction_score ~ C(languages)', data=lang_df_long).fit()
lang_anova_table = sm.stats.anova_lm(lang_model, typ=2)
print(lang_anova_table)
lang_p_value = lang_anova_table['PR(>F)'][0]
if lang_p_value < 0.05:
    print("\nConclusion: The p-value is extremely low. The choice of programming language has a STATISTICALLY SIGNIFICANT effect on job satisfaction.")

# ANOVA for Organization Size
print("\n--- ANOVA Results: Impact of Organization Size on Satisfaction ---")
org_df = df.dropna(subset=['OrgSize', 'job_satisfaction_score'])
org_model = ols('job_satisfaction_score ~ C(OrgSize)', data=org_df).fit()
org_anova_table = sm.stats.anova_lm(org_model, typ=2)
print(org_anova_table)
org_p_value = org_anova_table['PR(>F)'][0]
if org_p_value < 0.05:
    print("\nConclusion: The p-value is extremely low. Company size has a STATISTICALLY SIGNIFICANT effect on job satisfaction.")